# Independent m6A-Atlas Test Set Builder

It builds the final CSV files used for independent m6A-Atlas evaluation.

The final CSV schema is:

```text
seq,label
```


## Step 0. Project root and imports

Run this notebook from the repository root, or let the cell below find the root by searching for `build_independent_atlas_test_sets.py`.

In [3]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lamar_m6a").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "lamar_m6a").exists():
    raise RuntimeError("Cannot find project root containing lamar_m6a/. Run this notebook inside the project directory.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


PosixPath('/picb/rnasys2/lidaoqian/LAMAR-M6A')

In [5]:
from lamar_m6a.data.build_independent_atlas_test_sets import (
    build_intermediate_data,
    export_csv_files,
    default_data_path,
    default_output_dir,
)

print("Functions loaded.")


Functions loaded.


## Step 1. Configure paths

The GitHub-friendly default path is:

```text
data/raw/independent_atlas/{species Atlas table}
  -> data/intermediate/independent_atlas/{species}/atlas_{species}_a_full.data
  -> data/processed/independent_test_sets/atlas/full/{species}/*.csv
```


In [6]:
SPECIES = "rat"  # human, mouse, or rat
LENGTH = 801
NEGATIVE_RATIO = 10
SEED = 42

SPECIES_CONFIG = {
    "human": {
        "atlas_tables": [PROJECT_ROOT / "data" / "raw" / "independent_atlas" / "m6A-Atlas2_Human_Site.txt"],
        "gtf": PROJECT_ROOT / "data" / "raw" / "reference" / "Homo_sapiens.GRCh38.79.gtf.gz",
        "fasta": PROJECT_ROOT / "data" / "raw" / "reference" / "Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz",
    },
    "mouse": {
        "atlas_tables": [
            PROJECT_ROOT / "data" / "raw" / "independent_atlas" / "m6A-Atlas2_Mouse_Site_1.txt",
            PROJECT_ROOT / "data" / "raw" / "independent_atlas" / "m6A-Atlas2_Mouse_Site_2.txt",
        ],
        "gtf": PROJECT_ROOT / "data" / "raw" / "reference" / "Mus_musculus.GRCm38.102.gtf.gz",
        "fasta": PROJECT_ROOT / "data" / "raw" / "reference" / "Mus_musculus.GRCm38.dna.primary_assembly.fa.gz",
    },
    "rat": {
        "atlas_tables": [PROJECT_ROOT / "data" / "raw" / "independent_atlas" / "m6A-Atlas2_Rat_Site.txt"],
        "gtf": PROJECT_ROOT / "data" / "raw" / "reference" / "Rattus_norvegicus.Rnor_6.0.104.gtf.gz",
        "fasta": PROJECT_ROOT / "data" / "raw" / "reference" / "Rattus_norvegicus.Rnor_6.0.dna.toplevel.fa.gz",
    },
}

cfg = SPECIES_CONFIG[SPECIES]
ATLAS_TABLES = cfg["atlas_tables"]
GTF_PATH = cfg["gtf"]
FASTA_PATH = cfg["fasta"]
DATA_PATH = PROJECT_ROOT / default_data_path(SPECIES)
OUTPUT_DIR = PROJECT_ROOT / default_output_dir(SPECIES)

print("Species:", SPECIES)
print("Raw Atlas tables:", ATLAS_TABLES)
print("GTF:", GTF_PATH)
print("FASTA:", FASTA_PATH)
print("Intermediate .data:", DATA_PATH)
print("Final CSV output:", OUTPUT_DIR)


Species: rat
Raw Atlas tables: [PosixPath('/picb/rnasys2/lidaoqian/LAMAR-M6A/data/raw/independent_atlas/m6A-Atlas2_Rat_Site.txt')]
GTF: /picb/rnasys2/lidaoqian/LAMAR-M6A/data/raw/reference/Rattus_norvegicus.Rnor_6.0.104.gtf.gz
FASTA: /picb/rnasys2/lidaoqian/LAMAR-M6A/data/raw/reference/Rattus_norvegicus.Rnor_6.0.dna.toplevel.fa.gz
Intermediate .data: /picb/rnasys2/lidaoqian/LAMAR-M6A/data/intermediate/independent_atlas/rat/atlas_rat_a_full.data
Final CSV output: /picb/rnasys2/lidaoqian/LAMAR-M6A/data/processed/independent_test_sets/atlas/full/rat


## Step 2. Check input files

If you already have the old `.data` file, you may skip Step 3 and only run Step 4 after setting `DATA_PATH` to that file.

In [ ]:
for path in [*ATLAS_TABLES, GTF_PATH, FASTA_PATH]:
    print(path, "exists" if path.exists() else "MISSING")

print(DATA_PATH, "exists" if DATA_PATH.exists() else "not built yet")

## Step 3. Build intermediate `.data` from raw m6A-Atlas table

This builds the intermediate `.data` file from raw m6A-Atlas sites and species reference files.


In [ ]:
build_intermediate_data(
    atlas_tables=ATLAS_TABLES,
    gtf_path=GTF_PATH,
    fasta_path=FASTA_PATH,
    output_data=DATA_PATH,
    transcript_biotype="protein_coding",
    negative_ratio=NEGATIVE_RATIO,
    seed=SEED,
)

## Step 4. Export final per-experiment CSV files

This corresponds to the old notebook cell that used `WINDOW = 400` and wrote `{experiment}_merged.csv` files.

In [ ]:
summary = export_csv_files(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    length=LENGTH,
)
summary

# Independent m6A-Atlas Test Set Builder

It builds the final CSV files used for independent m6A-Atlas evaluation.

The final CSV schema is:

```text
seq,label
```


## Optional: export to another directory

Usually you do not need this. Use it only when you want to write a second copy of the final CSV files to a custom location.


## Step 6. Quick output check

Each output CSV should contain `seq` and `label`; for length 801, every sequence should be 801 nt.

In [ ]:
import pandas as pd

csv_files = sorted(OUTPUT_DIR.glob("*_merged.csv"))
print("CSV files:", len(csv_files))

if csv_files:
    example = csv_files[0]
    df = pd.read_csv(example)
    print("Example:", example)
    print(df.head())
    print(df["label"].value_counts())
    print(df["seq"].str.len().describe())